# Program 22
## Implementing the Viterbi Algorithm for POS Tagging
Given a sentence "Janet will back the bill", use the Viterbi algorithm to determine the most probable sequence of POS tags.

In [ ]:
import numpy as np

# Given Sentence
observation = ["Janet", "will", "back", "the", "bill"]

# Set of possible POS tags (States)
states = ['NNP', 'MD', 'VB', 'JJ', 'NN', 'RB', 'DT']
state2idx = {state: i for i, state in enumerate(states)}
idx2state = {i: state for i, state in enumerate(states)}

num_states = len(states)

# 1. Initial State Probabilities (pi)
# Assume Janet is highly likely to be NNP at the start
start_probabilities = np.zeros(num_states)
start_probabilities[state2idx['NNP']] = 1.0

# 2. Predefined Transition Probability Matrix (A)
# A[i][j] = P(State_j | State_i)
transition_matrix = np.full((num_states, num_states), 1e-4) # small smooth prob
# Define some strong transitions for our specific sentence
transition_matrix[state2idx['NNP'], state2idx['MD']] = 0.8
transition_matrix[state2idx['MD'], state2idx['VB']] = 0.8
transition_matrix[state2idx['MD'], state2idx['RB']] = 0.1
transition_matrix[state2idx['VB'], state2idx['DT']] = 0.6
transition_matrix[state2idx['DT'], state2idx['NN']] = 0.9

# Normalize to ensure valid probabilities
for i in range(num_states):
    transition_matrix[i, :] /= np.sum(transition_matrix[i, :])

# 3. Predefined Emission Probability Matrix B (as a dictionary of word given tag)
# B[state][word] = P(word | State)
emission_probs = {state: {} for state in states}
# Populate with some probabilities for the sentence words
emission_probs['NNP']['Janet'] = 0.5
emission_probs['MD']['will'] = 0.7
emission_probs['NN']['will'] = 0.1
emission_probs['VB']['back'] = 0.4
emission_probs['JJ']['back'] = 0.2
emission_probs['RB']['back'] = 0.2
emission_probs['NN']['back'] = 0.2
emission_probs['DT']['the'] = 0.8
emission_probs['NN']['bill'] = 0.6
emission_probs['VB']['bill'] = 0.1

def get_emission(state, word):
    return emission_probs[state].get(word, 1e-6) # small prob if unseen

# Viterbi Algorithm
def viterbi(obs, states, start_p, trans_p):
    n_obs = len(obs)
    n_states = len(states)
    
    viterbi_trellis = np.zeros((n_states, n_obs))
    backpointer = np.zeros((n_states, n_obs), dtype=int)
    
    # Initialization step
    word_0 = obs[0]
    for s in range(n_states):
        state_str = states[s]
        viterbi_trellis[s, 0] = start_p[s] * get_emission(state_str, word_0)
        backpointer[s, 0] = 0
        
    # Forward pass
    for t in range(1, n_obs):
        word_t = obs[t]
        for s in range(n_states):
            # Find the max probability path to this state 's' at time 't'
            state_str = states[s]
            max_prob = -1
            best_prev_state = -1
            
            for prev_s in range(n_states):
                prob = viterbi_trellis[prev_s, t-1] * trans_p[prev_s, s] * get_emission(state_str, word_t)
                if prob > max_prob:
                    max_prob = prob
                    best_prev_state = prev_s
                    
            viterbi_trellis[s, t] = max_prob
            backpointer[s, t] = best_prev_state
            
    # Termination step (find the best final state)
    best_path_prob = np.max(viterbi_trellis[:, n_obs - 1])
    best_final_state = np.argmax(viterbi_trellis[:, n_obs - 1])
    
    # Backtrace
    best_path = [best_final_state]
    for t in range(n_obs - 1, 0, -1):
        best_prev_state = backpointer[best_path[-1], t]
        best_path.append(best_prev_state)
        
    best_path.reverse() # Reverse to get the chronological sequence
    
    # Map indices back to state strings
    tag_sequence = [states[idx] for idx in best_path]
    
    return tag_sequence, best_path_prob

# Run Algorithm
predicted_tags, prob = viterbi(observation, states, start_probabilities, transition_matrix)

print("Sentence:", observation)
print("Predicted POS Tags:", predicted_tags)
print(f"Probability of this sequence: {prob:.4e}")